# O3 - features para previsão de procura (exploração passo a passo)

**O que o enunciado pede.**
1. Para cada balde (`PULocationID`, `day_of_week`, `hour_of_day`), calcular a média histórica do número de viagens, a variância e o valor de há 4 semanas (*lag*).
2. Calcular o *lag* com um **self-join** sobre datas de recolha desfasadas.
3. Calcular o índice de procura (`viagens atuais / média histórica`) e assinalar os pares zona-hora em que o índice passa de 2,0.
4. Juntar a tabela de zonas para anotar os resultados com o `service_zone`.

**A unidade de base** é o número de viagens de uma zona **numa hora concreta de um dia concreto**, por exemplo "Midtown Center, 2019-01-14, 8h". Cada balde (zona, dia da semana, hora) junta todas essas horas do mesmo tipo, como "Midtown Center, segundas, 8h", e a média histórica é a média delas.

**Porque é interessante para a otimização.** O self-join é feito sobre uma tabela grande: zonas × dias × 24 horas, ou seja milhões de linhas por ano. Isso dá shuffles grandes dos dois lados do join. O lag está no próprio enunciado como self-join, o que obriga a pensar em como o Spark executa esse join (sort-merge, co-particionamento).

| Bloco | O quê |
|---|---|
| O3-1 | viagens por (zona, dia, hora) |
| O3-2 | quantos dias de cada dia da semana tem o período |
| O3-3 | média e variância históricas por balde (com as horas a 0) |
| O3-4 | lag de 4 semanas (self-join) |
| O3-5 | índice de procura, flag > 2 e `service_zone` |
| O3-6 | plano e escrita |
| final | tudo junto numa função `run()` |

Uso **janeiro a março de 2019**. O lag de 4 semanas só existe a partir do 29º dia, e com 3 meses há uns 2 meses de lags.

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
         .appName("o3-dev")
         .config("spark.driver.memory", "1g")
         .config("spark.executor.memory", "1200m")
         .getOrCreate())

In [ ]:
import sys, contextlib, calendar, collections, datetime
from types import SimpleNamespace
from pyspark.sql import functions as F
sys.path.insert(0, "/project/src")
import taxi.m1_clean as m1

class NoProf:                      # profiler que não mede nada (o verdadeiro só no main.py)
    def profile(self, name):
        return contextlib.nullcontext()

a = SimpleNamespace(input="/project/data/raw", zones="/project/data/taxi_zone_lookup.csv",
                    output="/project/output", years=[2019], months="1-3", fraction=1.0)
trips, zones = m1.run(spark, a, set(), NoProf())

## Bloco O3-1: viagens por (zona, dia, hora)

1. **Só as viagens cujo timestamp cai no mês do ficheiro.** Os ficheiros têm algumas viagens com datas de outros meses ou anos (ex.: 2008, 2088). Aqui a data conta (é a chave do self-join), e uma viagem de 2008 criaria um dia isolado, sem lag nem histórico. São muito poucas viagens, e a escolha deve ficar no relatório.
2. `groupBy` por zona, data e hora → `trip_count`.
3. `day_of_week` com `F.dayofweek`: **1 = domingo**, …, 7 = sábado, como no M2.

O resultado `c` vai ser usado várias vezes. **Só no notebook** fica em cache.

In [ ]:
pickup = F.col("tpep_pickup_datetime")
t = trips.filter((F.year(pickup) == F.col("year")) & (F.month(pickup) == F.col("month")))

c = (t.groupBy("PULocationID", F.to_date(pickup).alias("pickup_date"), F.hour(pickup).alias("hour_of_day"))
      .agg(F.count("*").alias("trip_count"))
      .withColumn("day_of_week", F.dayofweek("pickup_date")))
c = c.cache()                      # só no notebook

print("linhas (zona, dia, hora) com viagens:", c.count())
c.orderBy(F.desc("trip_count")).show(5)

## Bloco O3-2: quantos dias de cada dia da semana tem o período

**O problema.** `c` só tem as horas **com** viagens. Uma zona com 3 viagens numa segunda às 4h e 0 nas outras segundas às 4h aparece com uma só linha. Se fizéssemos `avg(trip_count)`, a média dava 3, quando a média real (contando os zeros) é 3 / 13 ≈ 0,23.

**A solução, sem criar as linhas a 0.** Basta saber **quantos** dias de cada tipo há no período (`n_days`, por exemplo 13 segundas em jan–mar de 2019):

```
média     = Σx / N
variância = (Σx² − N · média²) / (N − 1)        (variância amostral)
```

Os zeros não somam nada a Σx nem a Σx². Só contam no N, e o N vem do calendário. É mais barato do que construir a grelha completa (zonas × dias × 24 horas), que no M3 era pequena mas aqui seria enorme.

O calendário faz-se em Python, porque é minúsculo: no máximo 7 linhas. `isoweekday()` dá segunda = 1 … domingo = 7, e `% 7 + 1` converte para a convenção do Spark (domingo = 1).

In [ ]:
dias = collections.Counter(
    datetime.date(y, m, d).isoweekday() % 7 + 1
    for y in a.years for m in m1.months(a.months)
    for d in range(1, calendar.monthrange(y, m)[1] + 1))
n_days = spark.createDataFrame(sorted(dias.items()), "day_of_week INT, n_days INT")
n_days.show()

## Bloco O3-3: média e variância históricas por balde

`groupBy` por balde (zona, dia da semana, hora) com Σx e Σx², junta-se o `n_days` (7 linhas, por isso broadcast) e aplicam-se as fórmulas.

A "média histórica" usa todo o período carregado, incluindo o próprio dia. É a leitura mais simples do enunciado. A alternativa seria usar só dias anteriores, com uma window.

A célula seguinte compara com o `avg` simples, que ignora os zeros, para se ver o erro que este passo evita.

In [ ]:
hist = (c.groupBy("PULocationID", "day_of_week", "hour_of_day")
         .agg(F.sum("trip_count").alias("sum_x"), F.sum(F.col("trip_count") ** 2).alias("sum_x2"))
         .join(n_days, "day_of_week")
         .withColumn("hist_mean", F.col("sum_x") / F.col("n_days"))
         .withColumn("hist_var", F.try_divide(F.col("sum_x2") - F.col("n_days") * F.col("hist_mean") ** 2,
                                              F.col("n_days") - 1))
         .drop("sum_x", "sum_x2"))

# 161 = Midtown Center, segundas (2), algumas horas
hist.filter((F.col("PULocationID") == 161) & (F.col("day_of_week") == 2)).orderBy("hour_of_day").show(6)

In [ ]:
# comparação: avg simples (sem os zeros) vs hist_mean, nas horas da madrugada, onde há mais zeros
ingenuo = c.groupBy("PULocationID", "day_of_week", "hour_of_day").agg(F.avg("trip_count").alias("avg_sem_zeros"))
(hist.join(ingenuo, ["PULocationID", "day_of_week", "hour_of_day"])
     .filter(F.col("hour_of_day") == 4)
     .withColumn("erro", F.col("avg_sem_zeros") / F.col("hist_mean"))
     .orderBy(F.desc("erro")).select("PULocationID", "day_of_week", "n_days", "avg_sem_zeros", "hist_mean", "erro").show(5))

## Bloco O3-4: lag de 4 semanas (self-join)

**O que é.** Para cada (zona, dia, hora), o número de viagens na mesma zona e hora **28 dias antes**. Como 28 dias são 4 semanas, é sempre o mesmo dia da semana.

**Como o enunciado pede: self-join sobre datas desfasadas.**
1. Faz-se uma cópia de `c` com a data **avançada** 28 dias (`date_add(pickup_date, 28)`) e a contagem renomeada para `lag_4w`. Nessa cópia, a linha de 2019-01-07 passa a dizer "2019-02-04".
2. Faz-se um `left join` de `c` com essa cópia em (zona, data, hora). Cada linha de 2019-02-04 encontra a de 2019-01-07.

**Lag em falta:**
- se o dia de há 28 dias **está no período carregado** mas não teve viagens nessa hora, o lag é **0**;
- se **não está no período** (os primeiros 28 dias, ou um mês que não foi carregado), o lag é **`null`**, porque não sabemos.

Para distinguir os dois casos, vê-se se o mês de `pickup_date − 28` está na lista de meses carregados (`year*12 + month`, o mesmo índice do M3).

**Limitação conhecida:** como em `c`, só se calcula o lag para as horas que **hoje** têm viagens. Uma hora com 0 viagens hoje e 40 há 4 semanas não aparece. Para o índice de procura isto não importa (0 viagens dá índice 0, nunca > 2).

In [ ]:
carregados = [y * 12 + m for y in a.years for m in m1.months(a.months)]
d28 = F.date_sub("pickup_date", 28)

lag = c.select("PULocationID", "hour_of_day",
               F.date_add("pickup_date", 28).alias("pickup_date"), F.col("trip_count").alias("lag_4w"))
f = (c.join(lag, ["PULocationID", "pickup_date", "hour_of_day"], "left")
      .withColumn("lag_4w", F.when((F.year(d28) * 12 + F.month(d28)).isin(carregados),
                                   F.coalesce("lag_4w", F.lit(0)))))

# uma zona e hora ao longo dos dias: o lag de cada linha é o trip_count de 28 dias antes
(f.filter((F.col("PULocationID") == 161) & (F.col("hour_of_day") == 8) & (F.col("day_of_week") == 2))
  .orderBy("pickup_date").select("pickup_date", "trip_count", "lag_4w").show(13))

## Bloco O3-5: índice de procura, flag e `service_zone`

```
demand_index = trip_count / hist_mean          (do balde: zona, dia da semana, hora)
high_demand  = demand_index > 2.0
```

- `hist_mean` nunca é 0 nas linhas de `f`, porque cada linha tem pelo menos 1 viagem, que entra na Σx do seu balde. Mesmo assim usamos `try_divide`, por segurança e coerência com o M3, já que o Spark 4 dá erro ao dividir por 0.
- Por fim junta-se a tabela de zonas para ter o `service_zone` (Yellow Zone, Boro Zone, Airports, EWR), além do nome e do borough.

In [ ]:
pz = zones.select(F.col("LocationID").alias("PULocationID"), F.col("Zone").alias("zone"),
                  F.col("Borough").alias("borough"), "service_zone")

res = (f.join(hist.drop("n_days"), ["PULocationID", "day_of_week", "hour_of_day"])
        .withColumn("demand_index", F.try_divide(F.col("trip_count"), F.col("hist_mean")))
        .withColumn("high_demand", F.coalesce(F.col("demand_index") > 2.0, F.lit(False)))
        .join(pz, "PULocationID"))

(res.groupBy("service_zone")
    .agg(F.count("*").alias("zona_horas"), F.sum(F.col("high_demand").cast("int")).alias("high_demand"))
    .withColumn("pct", F.round(100 * F.col("high_demand") / F.col("zona_horas"), 2)).show())

In [ ]:
# os maiores picos
(res.filter("high_demand").orderBy(F.desc("demand_index"))
    .select("zone", "service_zone", "pickup_date", "hour_of_day", "trip_count", "hist_mean", "lag_4w", "demand_index")
    .show(10, truncate=False))

### Cuidado: zonas com pouco volume

Numa zona-hora com média 0,2, basta 1 viagem para o índice dar 5. A célula seguinte mostra quantas flags vêm de baldes com média abaixo de 1 viagem por hora. **Decisão para o grupo** (como no M3): aceitar e documentar, ou exigir uma média mínima.

In [ ]:
(res.filter("high_demand").groupBy((F.col("hist_mean") < 1).alias("media_menor_que_1")).count().show())

## Bloco O3-6: plano e escrita

No plano procurar:
- o **self-join** (`SortMergeJoin` de `c` com a cópia desfasada), com um `Exchange` de cada lado, que são os shuffles grandes do O3;
- `InMemoryTableScan`: é o `c` em cache, e aparece três vezes (histórico, lado esquerdo do lag e lado direito do lag);
- `BroadcastHashJoin` com o `n_days` e com a tabela de zonas.

O resultado é escrito partido por `service_zone`.

In [ ]:
res.explain()

In [ ]:
res.write.mode("overwrite").partitionBy("service_zone").parquet("/project/output/o3")
spark.read.parquet("/project/output/o3").show(5, truncate=False)

## Versão final: tudo junto

Os blocos acima numa só função, com a mesma assinatura do M2 e do M3. É isto que iria para `src/taxi/o3_forecast.py`, e o `main.py` já sabe chamá-la (`"O3": "o3_forecast"`). No ficheiro, `m1` seria `from taxi import m1_clean as m1`.

**Diferença em relação ao notebook: sem cache.** No pipeline, `c` não está em cache, e o Spark **não** reutiliza o cálculo entre os três sítios onde `c` é usado. Confirmei-o no `explain()` com dados de teste: as viagens são lidas e agregadas **3 vezes** numa só escrita. É a baseline e é um bom primeiro bottleneck a medir: pôr `c` em cache (`--opt cache`) deve cortar isto para 1 leitura.

In [ ]:
def run(trips, zones, a, opt):
    spark = trips.sparkSession
    pickup = F.col("tpep_pickup_datetime")

    # 1) viagens por (zona, dia, hora); só timestamps dentro do mês do ficheiro
    c = (trips.filter((F.year(pickup) == F.col("year")) & (F.month(pickup) == F.col("month")))
              .groupBy("PULocationID", F.to_date(pickup).alias("pickup_date"), F.hour(pickup).alias("hour_of_day"))
              .agg(F.count("*").alias("trip_count"))
              .withColumn("day_of_week", F.dayofweek("pickup_date")))          # 1 = domingo

    # 2) média e variância por balde, contando as horas sem viagens (x = 0):
    #    média = Σx/N, var = (Σx² − N·média²)/(N−1), N = nº de dias desse dia da semana no período
    dias = collections.Counter(datetime.date(y, m, d).isoweekday() % 7 + 1          # domingo = 1, como o Spark
                               for y in a.years for m in m1.months(a.months)
                               for d in range(1, calendar.monthrange(y, m)[1] + 1))
    n_days = spark.createDataFrame(sorted(dias.items()), "day_of_week INT, n_days INT")
    hist = (c.groupBy("PULocationID", "day_of_week", "hour_of_day")
             .agg(F.sum("trip_count").alias("sum_x"), F.sum(F.col("trip_count") ** 2).alias("sum_x2"))
             .join(n_days, "day_of_week")
             .withColumn("hist_mean", F.col("sum_x") / F.col("n_days"))
             .withColumn("hist_var", F.try_divide(F.col("sum_x2") - F.col("n_days") * F.col("hist_mean") ** 2,
                                                  F.col("n_days") - 1))
             .drop("sum_x", "sum_x2", "n_days"))

    # 3) lag de 4 semanas: self-join com a data avançada 28 dias.
    #    dia de há 28 dias carregado mas sem viagens -> 0; fora do período -> null
    carregados = [y * 12 + m for y in a.years for m in m1.months(a.months)]
    d28 = F.date_sub("pickup_date", 28)
    lag = c.select("PULocationID", "hour_of_day",
                   F.date_add("pickup_date", 28).alias("pickup_date"), F.col("trip_count").alias("lag_4w"))
    f = (c.join(lag, ["PULocationID", "pickup_date", "hour_of_day"], "left")
          .withColumn("lag_4w", F.when((F.year(d28) * 12 + F.month(d28)).isin(carregados),
                                       F.coalesce("lag_4w", F.lit(0)))))

    # 4) índice de procura, flag > 2 e anotação com service_zone
    pz = zones.select(F.col("LocationID").alias("PULocationID"), F.col("Zone").alias("zone"),
                      F.col("Borough").alias("borough"), "service_zone")
    res = (f.join(hist, ["PULocationID", "day_of_week", "hour_of_day"])
            .withColumn("demand_index", F.try_divide(F.col("trip_count"), F.col("hist_mean")))
            .withColumn("high_demand", F.coalesce(F.col("demand_index") > 2.0, F.lit(False)))
            .join(pz, "PULocationID"))

    # 5) escrever: uma pasta por service_zone
    res.write.mode("overwrite").partitionBy("service_zone").parquet(f"{a.output}/o3")
    return res

final = run(trips, zones, a, set())
final.orderBy(F.desc("demand_index")).show(5, truncate=False)

In [ ]:
c.unpersist()